<a href="https://colab.research.google.com/github/sibandze/Bird-Intelligence-System/blob/dev-unsupervised/notebooks/exploration_and_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook: Data Loading + Spectogram Pipeline + Exploration

---



**Mount Google Drive & Define Directories**

In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Master Google Drive folder for dataset backups
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"Drive backup directory ready: {DRIVE_BACKUP_DIR}")

Mounted at /content/drive
Drive backup directory ready: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs


**Clone Repository & Install Dependencies**

In [16]:
import os
import sys

REPO_URL = "https://github.com/sibandze/Bird-Intelligence-System.git"
BRANCH = "dev-unsupervised"
REPO_DIR = "/content/Bird-Intelligence-System"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print("Repo exists. Pulling latest...")
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}  # this overwrites local changes

%cd {REPO_DIR}

if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt

Repo exists. Pulling latest...
/content/Bird-Intelligence-System
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 7.83 KiB | 616.00 KiB/s, done.
From https://github.com/sibandze/Bird-Intelligence-System
 * branch            dev-unsupervised -> FETCH_HEAD
   ef0d33a..ab6e7d8  dev-unsupervised -> origin/dev-unsupervised
HEAD is now at ab6e7d8 Update audio config requirements and filename signature
/content/Bird-Intelligence-System


Load Config & Restore Existing Files from Google Drive

In [4]:
import re
import shutil
import yaml

CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Extract relative paths from config
raw_audio_dir = config['data']['raw_audio_dir']             # data/raw_audio
processed_npy_dir = config['data']['processed_npy_dir']     # data/processed_spectrograms
metadata_dir = config['data']['metadata_dir']               # data/metadata

# Ensure target local directories exist inside the cloned repo
for rel_dir in [raw_audio_dir, processed_npy_dir, metadata_dir]:
    os.makedirs(os.path.join(REPO_DIR, rel_dir), exist_ok=True)

# Audio & Spectrogram Config Parameters to match against file naming schemes
audio_params = config.get('audio', {})
sr = audio_params.get('sr', 32000)
n_fft = audio_params.get('n_fft', 2048)
hop_length = audio_params.get('hop_length', 512)
n_mels = audio_params.get('n_mels', 128)

# Expected parameter string in processed file names (e.g., _sr32000_nfft2048_hop512_nmel128_)
spectrogram_pattern = f"sr{sr}_nfft{n_fft}_hop{hop_length}_nmel{n_mels}"

print(f"Scanning Drive backup using pattern key: '{spectrogram_pattern}'...\n")

Scanning Drive backup using pattern key: 'sr32000_nfft2048_hop512_nmel128'...



In [6]:
print(f"Contents of backup directory '{DRIVE_BACKUP_DIR}':")
backup_contents = os.listdir(DRIVE_BACKUP_DIR)

num_top_level_to_print = min(5, len(backup_contents))
for i in range(num_top_level_to_print):
    item_name = backup_contents[i]
    item_path = os.path.join(DRIVE_BACKUP_DIR, item_name)

    print(f"- {item_name}")

    if os.path.isdir(item_path):
        sub_contents = os.listdir(item_path)
        num_sub_level_to_print = min(3, len(sub_contents))
        for j in range(num_sub_level_to_print):
            print(f"  - {sub_contents[j]}")
        if len(sub_contents) > num_sub_level_to_print:
            print(f"  ... and {len(sub_contents) - num_sub_level_to_print} more.")

if len(backup_contents) > num_top_level_to_print:
    print(f"... and {len(backup_contents) - num_top_level_to_print} more top-level items.")

Contents of backup directory '/content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs':
- archived_audio
  - struthio_camelus_australis_audio.tar.gz
  - struthio_camelus_audio.tar.gz
  - struthio_molybdophanes_audio.tar.gz
  ... and 213 more.
- archived_spectrograms
  - struthio_camelus_australis_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
  - struthio_camelus_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
  - struthio_molybdophanes_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
  ... and 213 more.


In [7]:
import tarfile

def restore_files_from_drive(target_rel_dir, file_filter_fn):
    drive_dir = os.path.join(DRIVE_BACKUP_DIR, target_rel_dir)
    local_dir = os.path.join(REPO_DIR, target_rel_dir)

    restored_count = 0
    if os.path.exists(drive_dir):
        for filename in os.listdir(drive_dir):
            drive_file_path = os.path.join(drive_dir, filename)
            local_file_path = os.path.join(local_dir, filename)

            if os.path.isfile(drive_file_path) and file_filter_fn(filename):
                if not os.path.exists(local_file_path):
                    shutil.copy2(drive_file_path, local_file_path)
                    restored_count += 1

        print(f"[{target_rel_dir}] Restored {restored_count} matching file(s) from Drive.")
    else:
        print(f"[{target_rel_dir}] No Drive directory found at {drive_dir}")

def extract_archives_from_drive(archive_drive_sub_dir, local_target_sub_dir, file_filter_fn=None):
    """
    Extracts .tar.gz archives from a specified Google Drive subdirectory
    into a local target subdirectory within the repository.

    Args:
        archive_drive_sub_dir (str): The relative path within DRIVE_BACKUP_DIR
                                     where the .tar.gz archives are located (e.g., 'archived_audio').
        local_target_sub_dir (str): The relative path within REPO_DIR where
                                    the extracted files should be placed (e.g., 'data/raw_audio').
        file_filter_fn (callable, optional): A function to filter files during extraction.
                                            If None, all files are extracted.
    """
    drive_archive_path = os.path.join(DRIVE_BACKUP_DIR, archive_drive_sub_dir)
    local_extract_path = os.path.join(REPO_DIR, local_target_sub_dir)

    os.makedirs(local_extract_path, exist_ok=True) # Ensure local target directory exists

    extracted_count = 0
    archive_count = 0

    if os.path.exists(drive_archive_path):
        print(f"Scanning archive directory: {drive_archive_path}")
        for filename in os.listdir(drive_archive_path):
            if filename.endswith('.tar.gz'):
                archive_path = os.path.join(drive_archive_path, filename)
                print(f"  Extracting archive: {filename}")
                archive_count += 1
                try:
                    with tarfile.open(archive_path, "r:gz") as tar:
                        members_to_extract = []
                        if file_filter_fn:
                            for member in tar.getmembers():
                                if member.isfile() and file_filter_fn(member.name):
                                    members_to_extract.append(member)
                        else:
                            members_to_extract = [m for m in tar.getmembers() if m.isfile()]

                        for member in members_to_extract:
                            # Ensure no path traversal exploits, extract only the basename
                            member.name = os.path.basename(member.name)
                            # Only extract if the file does not already exist locally
                            if not os.path.exists(os.path.join(local_extract_path, member.name)):
                                tar.extract(member, path=local_extract_path)
                                extracted_count += 1

                except tarfile.ReadError as e:
                    print(f"Error reading tar.gz file {filename}: {e}")
                except Exception as e:
                    print(f"An unexpected error occurred while extracting {filename}: {e}")

        print(f"[{archive_drive_sub_dir}] Extracted {extracted_count} new file(s) from {archive_count} archive(s) to {local_extract_path}.")
    else:
        print(f"[{archive_drive_sub_dir}] No Drive archive directory found at {drive_archive_path}")


# 1. Restore Raw Audio (.ogg / .wav files) by extracting archives
extract_archives_from_drive(
    'archived_audio',
    raw_audio_dir,
    lambda f: f.endswith(('.ogg', '.wav', '.mp3'))
)

# 2. Restore Spectrograms matching current audio parameters by extracting archives
extract_archives_from_drive(
    'archived_spectrograms',
    processed_npy_dir,
    lambda f: f.endswith('.npy') and spectrogram_pattern in f
)

# 3. Restore Metadata files (.csv) - These are copied directly, not extracted from archives
restore_files_from_drive(
    metadata_dir,
    lambda f: f.endswith('.csv') and spectrogram_pattern in f
)


Scanning archive directory: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio
  Extracting archive: struthio_camelus_australis_audio.tar.gz


/tmp/ipykernel_973/2539018434.py:65: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=local_extract_path)


  Extracting archive: struthio_camelus_audio.tar.gz
  Extracting archive: struthio_molybdophanes_audio.tar.gz
  Extracting archive: rhea_americana_audio.tar.gz
  Extracting archive: rhea_americana_araneipes_audio.tar.gz
  Extracting archive: rhea_americana_albescens_audio.tar.gz
  Extracting archive: rhea_americana_intermedia_audio.tar.gz
  Extracting archive: rhea_americana_americana_audio.tar.gz
  Extracting archive: rhea_pennata_audio.tar.gz
  Extracting archive: rhea_pennata_pennata_audio.tar.gz
  Extracting archive: apteryx_australis_audio.tar.gz
  Extracting archive: apteryx_australis_lawryi_audio.tar.gz
  Extracting archive: apteryx_mantelli_audio.tar.gz
  Extracting archive: apteryx_rowi_audio.tar.gz
  Extracting archive: apteryx_owenii_audio.tar.gz
  Extracting archive: apteryx_owenii_owenii_audio.tar.gz
  Extracting archive: apteryx_haastii_audio.tar.gz
  Extracting archive: casuarius_casuarius_audio.tar.gz
  Extracting archive: casuarius_bennetti_audio.tar.gz
  Extracting ar

In [8]:
# Optional: Override/update runtime config parameters
# config['training']['batch_size'] = 32
# config['training']['epochs'] = 50
# config['logging']['use_wandb'] = False

RUN_CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config_colab_run.yaml")
with open(RUN_CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("\nConfiguration resolved and Drive files restored to local workspace.")


Configuration resolved and Drive files restored to local workspace.


In [9]:
# Execute the pipeline runner module with the updated config
!python -m pipeline.pipeline_runner --config configs/config_colab_run.yaml --full-dataset

✓ segment_seconds=3s -> 187 frames -> segment_size=200 (3.20s) [patch=25]
Starting Data Pipeline

Loading metadata...
🌐 Mode: Full Dataset (No balancing/class filtering)

Metadata summary
----------------
Total rows in source : 2,161
Selected classes     : 219
Target total samples : 2,161

Processing audio:   0% 0/2161 [00:00<?, ?file/s, fail=0, proc=0, skip=4]Error downloading https://xeno-canto.org/675445/download: 404 Client Error: Not Found for url: https://xeno-canto.org/675445/download
⚠️ Failed downloading XC675445
Processing audio:   8% 164/2161 [00:00<00:05, 344.74file/s, fail=1, proc=0, skip=198]Error downloading https://xeno-canto.org/396020/download: 404 Client Error: Not Found for url: https://xeno-canto.org/396020/download
⚠️ Failed downloading XC396020
Processing audio:  14% 298/2161 [00:01<00:06, 287.32file/s, fail=2, proc=0, skip=321]Error downloading https://xeno-canto.org/11910/download: 404 Client Error: Not Found for url: https://xeno-canto.org/11910/download
⚠️ Fa

In [12]:
import pandas as pd
import tarfile

# The pipeline runner generated a metadata file with a specific name.
config['data']['data_csv'] = f"metadata_full_{spectrogram_pattern}.csv"

# Construct the full path to the metadata file using the updated config
metadata_file_path = os.path.join(REPO_DIR, metadata_dir, config['data']['data_csv'])

# Load the metadata file
metadata_df = pd.read_csv(metadata_file_path)
display(metadata_df.head())

,common_name,scientific_name,Download_link,xc_id,scientific_name_id,rc_id,spectrogram_filename,local_spectrogram_path,total_frames,num_windows
0,Common Ostrich,Struthio camelus australis,https://xeno-canto.org/516153/download,XC516153,0,XC516153,XC516153_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,3349,14
1,Common Ostrich,Struthio camelus,https://xeno-canto.org/208209/download,XC208209,1,XC208209,XC208209_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,1672,7
2,Common Ostrich,Struthio camelus,https://xeno-canto.org/208128/download,XC208128,1,XC208128,XC208128_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,292,2
3,Common Ostrich,Struthio camelus,https://xeno-canto.org/46725/download,XC46725,1,XC46725,XC46725_sr32000_nfft2048_hop512_nmel128_seg200...,/content/Bird-Intelligence-System/data/process...,737,4
4,Common Ostrich,Struthio camelus,https://xeno-canto.org/673753/download,XC673753,1,XC673753,XC673753_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,729,4


Next, I will create a dictionary mapping each species to a list of its corresponding raw audio file paths.

In [13]:
# Define a dictionary to store audio file paths grouped by species
species_audio_files = {}

# Construct the full path to the raw audio directory
full_raw_audio_dir = os.path.join(REPO_DIR, raw_audio_dir)

for index, row in metadata_df.iterrows():
    species = row['scientific_name']
    filename = f'{row['rc_id']}.ogg' #row['filename'] # Assuming 'filename' is the column with audio filenames
    audio_file_path = os.path.join(full_raw_audio_dir, filename)

    if os.path.exists(audio_file_path):
        if species not in species_audio_files:
            species_audio_files[species] = []
        species_audio_files[species].append(audio_file_path)
    else:
        print(f"Warning: Audio file not found for {species}: {audio_file_path}")

print(f"Found {len(species_audio_files)} unique species with audio files.")
# Display a sample of the dictionary
for species, files in list(species_audio_files.items())[:3]:
    print(f"\nSpecies: {species}")
    for f in files[:5]: # Display up to 5 files per species for brevity
        print(f"  - {f}")

Found 216 unique species with audio files.

Species: Struthio camelus australis
  - /content/Bird-Intelligence-System/data/raw_audio/XC516153.ogg

Species: Struthio camelus
  - /content/Bird-Intelligence-System/data/raw_audio/XC208209.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC208128.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC46725.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC673753.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC563003.ogg

Species: Struthio molybdophanes
  - /content/Bird-Intelligence-System/data/raw_audio/XC292043.ogg


Finally, I will create a tar.gz archive for the audio files of each species. These archives will be saved in the Drive backup directory under a new `archived_audio` folder.

In [ ]:
# Create a directory for the archived audio files in the Drive backup
archive_output_dir = os.path.join(DRIVE_BACKUP_DIR, 'archived_audio')
os.makedirs(archive_output_dir, exist_ok=True)

for species, files in species_audio_files.items():
    # Sanitize species name for filename (replace problematic characters)
    sanitized_species = species.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').lower()
    archive_filename = f"{sanitized_species}_audio.tar.gz"
    archive_path = os.path.join(archive_output_dir, archive_filename)

    if not files:
        print(f"Skipping {species}: no audio files found.")
        continue

    with tarfile.open(archive_path, "w:gz") as tar:
        for f_path in files:
            # Add the file to the archive, preserving its base name
            tar.add(f_path, arcname=os.path.basename(f_path))
    print(f"Created archive for {species} at: {archive_path}")

print("\nAll species audio files have been archived.")

Created archive for Struthio camelus australis at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/struthio_camelus_australis_audio.tar.gz
Created archive for Struthio camelus at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/struthio_camelus_audio.tar.gz
Created archive for Struthio molybdophanes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/struthio_molybdophanes_audio.tar.gz
Created archive for Rhea americana at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/rhea_americana_audio.tar.gz
Created archive for Rhea americana araneipes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/rhea_americana_araneipes_audio.tar.gz
Created archive for Rhea americana albescens at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/rhea_americana_albescens_audio.tar.gz
Created archive for Rhea americana

In [ ]:
# Create a directory for the archived spectrogram files in the Drive backup
archive_spectrogram_output_dir = os.path.join(DRIVE_BACKUP_DIR, 'archived_spectrograms')
os.makedirs(archive_spectrogram_output_dir, exist_ok=True)

for species, _ in species_audio_files.items(): # Iterate through species using the keys from the audio files dictionary
    # Filter metadata_df for the current species to get all relevant spectrogram filenames
    species_df = metadata_df[metadata_df['scientific_name'] == species]

    spectrogram_files_to_archive = []
    for _, row in species_df.iterrows():
        spectrogram_filename = row['spectrogram_filename']
        spectrogram_file_path = os.path.join(REPO_DIR, processed_npy_dir, spectrogram_filename)
        if os.path.exists(spectrogram_file_path):
            spectrogram_files_to_archive.append(spectrogram_file_path)
        else:
            print(f"Warning: Spectrogram file not found for {species}: {spectrogram_file_path}")

    if not spectrogram_files_to_archive:
        print(f"Skipping {species}: no spectrogram files found.")
        continue

    # Sanitize species name for filename (replace problematic characters)
    sanitized_species = species.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').lower()
    archive_filename = f"{sanitized_species}_spectrograms_{spectrogram_pattern}.tar.gz"
    archive_path = os.path.join(archive_spectrogram_output_dir, archive_filename)

    with tarfile.open(archive_path, "w:gz") as tar:
        for f_path in spectrogram_files_to_archive:
            # Add the file to the archive, preserving its base name
            tar.add(f_path, arcname=os.path.basename(f_path))
    print(f"Created archive for {species} at: {archive_path}")

print("\nAll species spectrogram files have been archived.")

Created archive for Struthio camelus australis at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/struthio_camelus_australis_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Struthio camelus at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/struthio_camelus_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Struthio molybdophanes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/struthio_molybdophanes_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Rhea americana at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/rhea_americana_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Rhea americana araneipes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/rhea_americana_araneipes_spectrograms_sr32000_nf

In [14]:
metadata_source_path = metadata_file_path
drive_metadata_backup_dir = os.path.join(DRIVE_BACKUP_DIR, metadata_dir)
os.makedirs(drive_metadata_backup_dir, exist_ok=True)
metadata_destination_path = os.path.join(drive_metadata_backup_dir, os.path.basename(metadata_source_path))
shutil.copy2(metadata_source_path, metadata_destination_path)

print(f"Backed up metadata file to: {metadata_destination_path}")

Backed up metadata file to: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/data/metadata/metadata_full_sr32000_nfft2048_hop512_nmel128.csv


## Running Experiments
Now, let's run some experiments using the `experiment_runner.py` script. We'll use the `config_colab_run.yaml` generated previously and save the results directly to Google Drive.

In [17]:
# Define the results directory to be inside the Drive backup for persistence
RESULTS_DIR_DRIVE = os.path.join(DRIVE_BACKUP_DIR, 'experiment_results')
os.makedirs(RESULTS_DIR_DRIVE, exist_ok=True)

# Run the experiment_runner.py script
# We use the config_colab_run.yaml which includes any Colab-specific overrides.
# The --suite 'quick_baseline' is a good starting point for a quick run.
# The --results-dir points to the Google Drive location.
print(f"Running experiments. Results will be saved to: {RESULTS_DIR_DRIVE}")
!python -m experiments.experiment_runner \
    --config {RUN_CONFIG_PATH} \
    --suite quick_baseline \
    --results-dir {RESULTS_DIR_DRIVE}


Running experiments. Results will be saved to: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/experiment_results
✓ segment_seconds=3s -> 187 frames -> segment_size=200 (3.20s) [patch=25]

🚀 Running Experiment Suite: quick_baseline

📊 Total configurations to run: 5
⚠️  This will take approximately 50 minutes (assuming ~10 min/run)
💾 Results will be saved to: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/experiment_results

✓ Loaded 2139 samples from /content/Bird-Intelligence-System/data/metadata/metadata_full_sr32000_nfft2048_hop512_nmel128.csv

────────────────────────────────────────────────────────────────────────────────
📋 Sweep: baseline_lr_sweep
   Description: Sweep over learning rates for baseline model
────────────────────────────────────────────────────────────────────────────────
   Configurations: 5


  [0] Training: {'learning_rate': 1e-05}
      ✗ Error during training: The least populated class in y has only 1 member, which is too few